# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf

# Централизованная настройка логгера
from src.utils.logger import setup_logging


setup_logging()

# Находим корень проекта
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Инициализируем Hydra
from src.utils.notebook_setup import init_nlp_notebook #noqa E402


cfg = init_nlp_notebook()

if "paths" not in cfg:
    cfg.paths = OmegaConf.create()
cfg.paths.data_dir = str(PROJECT_ROOT / "data")

device = "cuda" if torch.cuda.is_available() else "cpu"

NLP Environment ready. Root: c:\nlp_template_decoder


# Data & Tokenizer

In [2]:
from src.core.data.builder import NLPDataModule


# 1. Загрузка токенизатора
tokenizer = instantiate(cfg.model.tokenizer).build()

# ВАЖНО: Для батчевой генерации (inference) паддинг должен быть слева
tokenizer.padding_side = "left"

# 2. Подготовка DataModule
datamodule = NLPDataModule(data_cfg=cfg.data, tokenizer=tokenizer)
datamodule.prepare_data()
datamodule.setup(stage="validate")

val_dataloader = datamodule.val_dataloader()
sample_batch = next(iter(val_dataloader))

print("Input IDs shape:", sample_batch["input_ids"].shape)
if "labels" in sample_batch:
    print("Labels contain -100:", (sample_batch["labels"] == -100).any().item())

c:\nlp_template_decoder\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:src.core.models.tokenization:Загрузка токенизатора: HuggingFaceM4/tiny-random-LlamaForCausalLM
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got -1. This may result in unexpected behavior.
INFO:src.core.data.builder:Нашли кэш обработанных данных: c:\nlp_template_decoder\data/processed\sft_dataset_processed_b79be0ab. Подготовка пропущена.


Input IDs shape: torch.Size([8, 166])
Labels contain -100: True


c:\nlp_template_decoder\.venv\lib\site-packages\transformers\tokenization_utils_base.py:2368: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


# Base Model init

In [ ]:
from src.core.models.generator import HFTextGenerator
from src.core.prompts.manager import PromptManager


# 1. Загружаем базовую модель через билдер
model_builder = instantiate(cfg.model.builder, tokenizer=tokenizer)
model = model_builder.build()
model.eval()

print(f"Base Model: {model.config._name_or_path}")

# 2. Инициализируем генератор
# generation_kwargs лежат в корне cfg, согласно main.yaml
generator = HFTextGenerator(
    model=model,
    tokenizer=tokenizer,
    generation_kwargs=cfg.generation_kwargs,
    cleaner_cfg=cfg.model.get("cleaner")
)

# 3. Инициализируем менеджер промптов, загружая шаблоны из configs/prompts/default.yaml
prompt_manager = PromptManager(templates=OmegaConf.to_container(cfg.prompts, resolve=True))

INFO:src.core.models.builder:Загрузка базовой архитектуры: HuggingFaceM4/tiny-random-LlamaForCausalLM
INFO:src.core.models.builder:Применение квантизации BitsAndBytes.
[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 21/21 [00:00<00:00, 1105.30it/s]
INFO:src.core.models.builder:Активация Gradient Checkpointing (Экономия VRAM).
INFO:src.core.models.builder:Режим PEFT: Инициализация нового LoRA адаптера.
INFO:src.core.models.builder:LoRA: 11,776 обучаемых из 1,044,048 (1.1279%)
INFO:src.core.prompts.manager:Инициализирован PromptManager. Загружено шаблонов: 3


Base Model: HuggingFaceM4/tiny-random-LlamaForCausalLM


# Forward Pass (Loss) vs Generation

In [ ]:
import torch
from hydra.utils import instantiate


sample_batch = {k: v.to(device) for k, v in sample_batch.items() if isinstance(v, torch.Tensor)}

# 1. Проверка режима "Обучение" (Teacher Forcing для подсчета Perplexity)
with torch.no_grad():
    outputs = model(
        input_ids=sample_batch["input_ids"],
        attention_mask=sample_batch["attention_mask"],
        labels=sample_batch.get("labels")
    )
    perplexity = torch.exp(outputs.loss).item() if outputs.loss is not None else float('nan')
    print(f"Zero-shot Baseline Perplexity (Loss): {perplexity:.2f}")

# 2. Проверка режима "Генерация" (Inference)
# Поскольку текстовые колонки удалены из datamodule, берем 2 реальных текста из сырого датасета
fetcher = instantiate(cfg.data.source)
raw_dataset = fetcher.load()
# Если сплита validation изначально нет, возьмем из train
raw_split = raw_dataset["validation"] if "validation" in raw_dataset else raw_dataset["train"]

text_col = cfg.data.get("prompt_column") or cfg.data.get("text_column")
raw_texts = raw_split.select(range(2))[text_col]

# Рендерим промпты через PromptManager (например, шаблон 'summarization')
# Если датасет просто текст, можно использовать заглушку
try:
    prompts = [prompt_manager.render("summarization", text=text) for text in raw_texts]
except ValueError:
    prompts = raw_texts # Fallback, если шаблон не подходит

# Генерируем ответы одной строкой
responses = generator.generate(texts=prompts, max_new_tokens=50, do_sample=False)

print("\n--- ZERO-SHOT GENERATION EXAMPLE ---")
for p, r in zip(prompts, responses): #noqa B905
    print(f"PROMPT:\n{p}")
    print(f"\nRESPONSE:\n{r}")
    print("-" * 50)

INFO:src.core.data.fetcher:HF датасет найден локально: c:\nlp_template_decoder\data/raw\HuggingFaceH4_testing_alpaca_small
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Zero-shot Baseline Perplexity (Loss): 32033.38


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



--- ZERO-SHOT GENERATION EXAMPLE ---
PROMPT:
Сделай краткое содержание следующего текста на русском языке.
Выдели ключевые факты и сохрани структуру.

Текст: What is a digital identity and why is it important?

Краткое содержание:

RESPONSE:
logging Société заняめ диphysovis разли superfickunftüllktóbernpm userloyсобam sod passengers gesch Comment też wurdenWSbon phr тя Publicterm cruelanche svensk automatic przeciไ Review course BodAtIndexskyalesumbnailstenagersRotlb combinedarel doubles Whit
--------------------------------------------------
PROMPT:
Сделай краткое содержание следующего текста на русском языке.
Выдели ключевые факты и сохрани структуру.

Текст: Guess the next word

The family decided to

Краткое содержание:

RESPONSE:
logging Sociétéfluttercontribovis разли superfickunftüllktóbernpm userloyсобam sod passengers gesch Comment też wurdenWSbon phr тя Publicterm cruelanche svensk automatic przeciไ Review course BodAtIndexskyalesumbnailstenagersRotlb combinedarelájaбер Poli

# Baseline Metrics

In [6]:
from hydra.utils import instantiate
from torchmetrics.text.rouge import ROUGEScore
from tqdm.auto import tqdm


rouge_metric = ROUGEScore()

all_preds = []
all_refs = []

# Вытаскиваем нужные колонки (с дефолтным fallback на 'completion', как в конфигах)
text_col = cfg.data.get("prompt_column") or cfg.data.get("text_column")
target_col = cfg.data.get("target_column", "completion")

# Поскольку текстовые колонки удалены из datamodule, загружаем сырой датасет
fetcher = instantiate(cfg.data.source)
raw_dataset = fetcher.load()

# Берем валидационный сплит (или train, если валидации изначально нет)
raw_split = raw_dataset["validation"] if "validation" in raw_dataset else raw_dataset["train"]

# Берем 50 сэмплов из сырой валидации
eval_samples = raw_split.select(range(min(50, len(raw_split))))

eval_prompts = []
for row in eval_samples:
    try:
        eval_prompts.append(prompt_manager.render("summarization", text=row[text_col]))
    except ValueError:
        eval_prompts.append(row[text_col])

all_refs = [row[target_col] for row in eval_samples]

# Генерируем предсказания батчами
batch_size = 8
for i in tqdm(range(0, len(eval_prompts), batch_size), desc="Zero-shot Generation"):
    batch_prompts = eval_prompts[i : i + batch_size]

    batch_preds = generator.generate(texts=batch_prompts, max_new_tokens=50)
    all_preds.extend(batch_preds)

# Считаем ROUGE
rouge_metric.update(all_preds, all_refs)
results = rouge_metric.compute()

print("\n--- Baseline ROUGE Scores ---")
for metric_name, tensor_val in results.items():
    print(f"{metric_name}: {tensor_val.item():.4f}")

INFO:src.core.data.fetcher:HF датасет найден локально: c:\nlp_template_decoder\data/raw\HuggingFaceH4_testing_alpaca_small
Zero-shot Generation: 100%|██████████| 7/7 [00:07<00:00,  1.02s/it]



--- Baseline ROUGE Scores ---
rouge1_fmeasure: 0.0023
rouge1_precision: 0.0026
rouge1_recall: 0.0028
rouge2_fmeasure: 0.0000
rouge2_precision: 0.0000
rouge2_recall: 0.0000
rougeL_fmeasure: 0.0023
rougeL_precision: 0.0026
rougeL_recall: 0.0028
rougeLsum_fmeasure: 0.0023
rougeLsum_precision: 0.0026
rougeLsum_recall: 0.0028
